# Exploring hotel data with pandas

Imagine helping a visitor find a hotel: **can we find a room below EUR 100 per night with a guest rating of at least 4 out of 5?**

A data scientist starts with questions, not a list of commands. Before making a recommendation, we will ask:

1. **What hotels and information do we actually have?**
2. **What does a typical night cost, and how much do prices vary?**
3. **Does every hotel have a rating from both review sources?**
4. **Which hotels meet our budget and guest-rating requirements?**
5. **Which qualifying hotels are cheapest, and what else should we consider?**

Each question leads to a small piece of code and an answer we can read from its output. Along the way, we learn how to load, inspect, summarise, filter, and sort a DataFrame.

This notebook starts from a local CSV. You do **not** need to complete `01_pandas_basics.ipynb` first.

## Before we start: a table in Python

**pandas** is a Python library for working with tables. A **DataFrame** is a table with labelled rows and columns. In our example, each row is one hotel's price quote; columns contain information such as its price and guest rating.

Use the course Python environment and run the cells from top to bottom. The small CSV is already included in the repository; no download or previous notebook output is needed.

The setup below lets us open the same file from either the course's main folder or this notebook's folder. It is just file-location setup; our data exploration begins with `read_csv`.

In [ ]:
# pd is the short name we use to access pandas.
import pandas as pd
from pathlib import Path

# Find the supplied CSV when running from the course's main folder.
data_file = Path("lectures/lecture04-pandas-basics/hotel_vienna_restricted.csv")

# Notebook editors may instead start in the folder containing this notebook.
if not data_file.exists():
    data_file = Path("hotel_vienna_restricted.csv")

## 1. What hotels and information do we actually have?

We use an already prepared sample from the [Hotels Europe teaching data](https://gabors-data-analysis.com/datasets/hotels-europe/). It contains one-night weekday quotes for the **Vienna search area in November 2017**, including some nearby municipalities. The sample contains hotels with star categories from 3 to 4, including 3.5.

These are historical teaching data, not current offers. The preparation and selection have already been done for us; this is not a complete or representative list of hotels.

First, load the CSV into a DataFrame called `hotels`. Reading a file does not change the file on disk.

In [ ]:
# Read the local CSV into a table in Python's memory.
hotels = pd.read_csv(data_file)

# Backup if the local CSV is missing (requires an internet connection):
# Comment out the local read above, then uncomment BOTH lines below and rerun.
# url = "https://raw.githubusercontent.com/ulrichwohak/Coding-1-Introduction-to-Python/main/lectures/lecture04-pandas-basics/hotel_vienna_restricted.csv"
# hotels = pd.read_csv(url)

### Keep the columns relevant to our questions

The CSV contains more information than we need today. We select eight columns to keep the display readable:

| Column | Meaning |
| --- | --- |
| `hotel_id` | Hotel identifier, not a quantity to average |
| `neighbourhood` | Area label recorded in the data |
| `stars` | Hotel star category; different from a guest rating |
| `price` | Quoted price in EUR for this one-night stay |
| `ratings` | Average guest rating on a 0–5 scale |
| `rating_count` | Number of reviews behind that guest rating |
| `distance` | Distance from the main city centre, in miles |
| `ratingta` | Alternative guest rating from Tripadvisor |

`hotels["price"]` selects one column as a **Series**. `hotels[["price", "ratings"]]` selects a **list of columns** and returns a DataFrame. The inner brackets make the list; the outer brackets select from the table.

In [ ]:
# Select a list of columns; the original CSV still contains all its columns.
columns = [
    "hotel_id", "neighbourhood", "stars", "price",
    "ratings", "rating_count", "distance", "ratingta",
]
hotels = hotels[columns]

# Preview the first five rows, not the entire dataset.
hotels.head()

Read across one row: which value is the price, and which is the guest rating?

The numbers at the far left are the **index**, or row labels. They are not the `hotel_id` column. Later, filtered tables will retain their original row labels.

`head()` is a **method**: an operation we ask the DataFrame to perform. Parentheses call the method; `head(3)` would request three rows instead of five.

In [ ]:
# shape gives (number of rows, number of columns); no parentheses are needed.
hotels.shape

In [ ]:
# Inspect column names, non-missing counts, and the types pandas read.
hotels.info()

We have 217 rows and the eight columns we selected. In this prepared sample there is one row per hotel.

In `info()`, `int64` and `float64` are numeric types; `object` here contains text. A numeric type does not make every column a useful measurement: `hotel_id` is still just an identifier.

Before treating these hotels as one group, check the mix of star categories. Which category is most common?

In [ ]:
# Count how many rows belong to each star category.
hotels["stars"].value_counts()

## 2. What does a typical night cost, and how much do prices vary?

Is EUR 100 a generous budget or a restrictive one in this sample? `describe()` gives a quick numerical overview. Select meaningful measurements rather than averaging identifiers or calendar fields.

In [ ]:
# Summarise prices, guest ratings, and distance to the centre.
hotels[["price", "ratings", "distance"]].describe()

Read the **price** column first:

- `count`: how many non-missing prices contribute to the summary.
- `mean`: the average price.
- `min` and `max`: the lowest and highest prices.
- `50%`: the median, or middle price when the prices are ordered.

Here, the median is EUR 100, the mean is about EUR 109.72, and the range is EUR 50–383. A single average therefore does not tell us everything about the available prices.

`25%` and `75%` are other percentile cutoffs; `std` describes spread. Today, focus on finding and reading the counts, typical values, and range, not on calculating statistics.

**Discuss:** Our rule will be *below* EUR 100. Should a hotel costing exactly EUR 100 qualify?

## 3. Does every hotel have a rating from both review sources?

A blank rating does not mean a rating of zero. Before comparing reviews, check whether information is missing.

In [ ]:
# isna() marks missing entries True and observed entries False.
missing_ratings = hotels[["ratings", "ratingta"]].isna()

# sum() counts the True entries separately for each column.
missing_ratings.sum()

The primary `ratings` column is complete in this prepared sample. Three alternative `ratingta` values are missing.

We do not replace missing ratings with zero or discard these hotels automatically. Our search uses the primary guest rating, which is available. If we wanted to compare the two review sources, those three missing values would matter.

This also explains why `describe()` reports a separate non-missing `count` for each column.

## 4. Which hotels meet our budget and guest-rating requirements?

Start with one condition: a price below EUR 100. Comparing a column with a number checks **every row**, so we do not need to write a loop.

In [ ]:
# Each row gets True if its price is strictly below 100, otherwise False.
within_budget = hotels["price"] < 100
within_budget.head()

Use that sequence of True/False values to choose rows. Read `.loc[rows, columns]` as: **which rows, then which columns?**

With only `.loc[within_budget]`, we select the matching rows and keep every column.

In [ ]:
# Keep the rows marked True; hotels itself is not changed.
affordable = hotels.loc[within_budget]
affordable.head()

Now require a guest rating of at least 4 as well. For pandas columns, `&` means both conditions must hold. Put each comparison in parentheses; do not use Python's `and` between two Series.

In [ ]:
# A hotel must pass both tests, row by row.
good_value = (hotels["price"] < 100) & (hotels["ratings"] >= 4)

# Select matching rows and only the information needed for our shortlist.
shortlist = hotels.loc[
    good_value,
    ["hotel_id", "price", "ratings", "rating_count", "distance"],
]
shortlist.shape

There are **51 qualifying hotels** in this sample. `shortlist` is a new result; we have not removed the other rows from `hotels` or changed the CSV.

We chose the name `good_value` for our two selection criteria. It is not proof that these hotels are bargains: the price, rating, location, and number of reviews may point in different directions.

## 5. Which qualifying hotels are cheapest, and what else should we consider?

A filter selects rows; a sort puts those rows in a useful order. Show the five cheapest options that meet our criteria.

In [ ]:
# sort_values() returns a sorted table; assigning it keeps that result.
shortlist = shortlist.sort_values("price")

# The default order is ascending, so the lowest prices appear first.
shortlist.head(5)

The cheapest qualifying quote costs **EUR 60**, with a guest rating of **4.1** and a distance of **1.1 miles** from the centre.

**Discuss:** Would you automatically choose the cheapest hotel? Compare the distances and review counts. A shortlist supports a decision; it does not make the decision for us.

Notice that the row labels did not restart at zero. Sorting changed the order of rows, not the hotels' identifiers.

## Main lesson complete

We went from a question to a local file, then to an inspected table and a reproducible shortlist. Someone else can rerun our cells and see exactly which criteria we used.

**Try a variation:** A different visitor cares most about location. Find hotels below EUR 120 with a guest rating of at least 4, but show the five **closest to the centre**, rather than the cheapest. Which part of the filter changes? Which part of the sorting changes?

The remaining sections are optional extensions. The main lesson does not depend on them.

## Optional A. How far away are these hotels in kilometres?

The source reports miles. We can create a new column by applying the same calculation to a whole column. One mile is 1.609344 kilometres.

In [ ]:
# Make an independent copy before adding information to the selected table.
shortlist = shortlist.copy()

# Multiply every distance by the conversion factor; no loop is needed.
shortlist["distance_km"] = shortlist["distance"] * 1.609344
shortlist[["hotel_id", "price", "distance_km"]].head()

## Optional B. Do higher-star hotels have higher typical prices here?

Use the full sample, not just our shortlist. `groupby("stars")` separates rows by star category; selecting `price` and calling `median()` summarises prices within each category.

In [ ]:
# Calculate one median price for each star category in the full sample.
median_price_by_stars = hotels.groupby("stars")["price"].median()
median_price_by_stars

The medians are EUR 84, EUR 100, and EUR 117 for the 3-, 3.5-, and 4-star groups. That describes these selected hotels; it does not show that changing a hotel's star category would cause its price to change.

## Optional C. How can we share the shortlist as a file?

A DataFrame in memory is not yet a saved file. We can export a new CSV without overwriting our input. The `scratch` folder below is created inside the current working directory and is ignored by Git in this course repository.

In [ ]:
# Create the output folder if it does not already exist.
Path("scratch").mkdir(exist_ok=True)

# Save the shortlist, without adding the pandas row labels as a CSV column.
shortlist.to_csv("scratch/lecture04_hotel_shortlist.csv", index=False)

# Display the full path so we know where the new file was saved.
Path("scratch/lecture04_hotel_shortlist.csv").resolve()

## Finding help and sources

You do not need to memorise every DataFrame method. These official pages let you check what a method accepts and returns:

- [Load a CSV: `read_csv`](https://pandas.pydata.org/pandas-docs/version/2.3/reference/api/pandas.read_csv.html)
- [Read numerical summaries: `DataFrame.describe`](https://pandas.pydata.org/pandas-docs/version/2.3/reference/api/pandas.DataFrame.describe.html)
- [Select rows and columns: `DataFrame.loc`](https://pandas.pydata.org/pandas-docs/version/2.3/reference/api/pandas.DataFrame.loc.html)
- [Choose the sort order: `DataFrame.sort_values`](https://pandas.pydata.org/pandas-docs/version/2.3/reference/api/pandas.DataFrame.sort_values.html)

The prepared CSV was already supplied with this course. For the original data definitions and educational-use terms, see the [Hotels Europe dataset description](https://gabors-data-analysis.com/datasets/hotels-europe/). For attribution of the adapted course materials, see [NOTICE.md](../../NOTICE.md).